# Coming from Brightway

You know Brightway: projects, databases, activities, exchanges, `bw2calc.LCA` and methods as tuples. This notebook does the same LCA tasks twice, once in Brightway and once in headwater, on the toy T-shirt. After each pair it says in a few sentences what differs and why.

Both tools must give the same T-shirt: 1.19205 kg CO₂-eq. The notebook checks it.

| Task | Brightway | headwater |
|---|---|---|
| 1. Set up | a project in your Brightway folder | a cache folder for releases, and database folders in git |
| 2. Import the background | `bw2io` importers, then `Database.write` | `headwater release import`, once per version, then `hw.load` |
| 3. Find an activity | `Database.search`, `get_node` | `bg.find` |
| 4. One footprint | `bc.LCA(...)`, `lci()`, `lcia()` | `db.calc(bg, "tshirt:tshirt")` |
| 5. Every footprint | `lci(factorize=True)`, then `redo_lcia` per activity, or `MultiLCA` | already solved by `hw.load`; `hw.score_many` |
| 6. Contribution analysis | `lca.characterized_inventory`, bw2analyzer | `result.where()`, `result.contributions()` |
| 7. Change one input and recompute | edit the exchange, `save()`, a new `LCA` | edit one row of `exchanges.csv`, `calc` again; `result.sensitivity()` predicts it |
| 8. Import an Excel inventory | `bw2io.ExcelImporter` | `headwater database import-excel` |
| 9. Import a Brightway project | the data is already in the project | `headwater database import-brightway` |

**Versions.** The Brightway side uses bw2data 4.5.4, bw2calc 2.3, bw2io 0.9.14 and peewee 3.19.0, the versions of [bench/README.md](../bench/README.md). They are the `brightway` dependency group of this repository, kept apart so that installing headwater does not install Brightway. Install both groups to run this notebook:

```bash
uv sync --group notebooks --group brightway
```

Without Brightway, the notebook tests skip this notebook. [Notebook 05](05-brightway-and-headwater-on-ecoinvent.ipynb) does the same tasks on ecoinvent 3.12, timed.

## 1. Set up

**Brightway.** Brightway keeps everything in one folder, named by the environment variable `BRIGHTWAY2_DIR` or a default per user. Importing `bw2data` opens that folder and migrates it to its own version, whatever is in it. So the first line of this cell points `BRIGHTWAY2_DIR` at a new, empty temporary folder, in the same cell and before the import. Your own Brightway projects are never opened.

Brightway prints log lines with the time of day and progress bars. The cell routes its log through Python's `logging` and silences it, turns the progress bars off, and hides one warning about an optional solver. Without this the notebook's output would change on every run. In your own work, leave them on.

In [1]:
import contextlib
import io
import logging
import os
import shutil
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path

import numpy as np

work = Path(tempfile.mkdtemp(prefix="brightway-and-headwater-")).resolve()
(work / "brightway").mkdir()
os.environ["BRIGHTWAY2_DIR"] = str(work / "brightway")  # never your own Brightway folder
os.environ["BRIGHTWAY_NO_STRUCTLOG"] = "1"  # log through Python's logging, so it can be silenced
os.environ["TQDM_DISABLE"] = "1"  # no progress bars
logging.getLogger("brightway-stdout-feedback").addFilter(lambda record: False)
warnings.filterwarnings("ignore", module="bw2calc")  # the optional scikit-umfpack solver

import bw2calc as bc  # noqa: E402
import bw2data as bd  # noqa: E402
import bw2io as bi  # noqa: E402

bd.projects.set_current("tshirt")
assert Path(bd.projects.dir).resolve().is_relative_to(work)
print("Brightway project:", bd.projects.current)

Brightway project: tshirt


**headwater.** headwater has no project. Background releases sit in a cache folder, `HEADWATER_HOME`, and are never edited. Your databases are ordinary folders, usually in a git repository. Here `HEADWATER_HOME` is a temporary folder too, so your own cache is not touched.

The helper `headwater(...)` runs the `headwater` command as in a terminal and prints what it prints, with the temporary folder shown as `<work>`.

In [2]:
import headwater as hw

os.environ["HEADWATER_HOME"] = str(work / "headwater-home")
here = Path.cwd()
repo = next((p for p in [here, *here.parents] if (p / "src/headwater/demo").is_dir()), None)
assert repo, "run this notebook from a checkout of the headwater repository"
demo = repo / "src/headwater/demo"
HEADWATER = shutil.which("headwater", path=str(Path(sys.executable).parent))


def headwater(*args):
    """Run the headwater command and print what it prints."""
    done = subprocess.run([HEADWATER, *map(str, args)], capture_output=True, text=True, cwd=work)
    text = (done.stdout + done.stderr).replace(str(work), "<work>").replace(str(repo), "<repo>")
    print(text.rstrip())
    if done.returncode:
        print(f"(exit code {done.returncode})")

**What differs.** A Brightway project is a place where you import, edit and calculate. Two notebooks that use two projects each import their own copy of ecoinvent. In headwater the background is shared and read-only: every database on the machine reads the same release, and your own database is a few text files next to your other work.

## 2. Import the background

The toy background has three processes: electricity (0.5 kg CO₂ and 0.001 kg CH₄ per kWh), cotton fabric (3 kWh, 2.0 kg CO₂ and 0.004 kg CH₄ per kg) and the incineration of textile waste (1.2 kg CO₂ per kg treated). The method counts CO₂ as 1 and CH₄ as 29.8.

**Brightway.** For ecoinvent you would run bw2io's ecoSpold2 importers. The toy is three processes, so here it is written with `Database.write`, the way bw2io writes in the end. The waste treatment follows ecoinvent's convention: its production amount is -1 kg. Then the method, as a tuple and a list of `(flow, factor)`.

In [3]:
bd.Database("toy-biosphere").write(
    {
        ("toy-biosphere", "co2"): {
            "name": "Carbon dioxide, fossil",
            "unit": "kilogram",
            "type": "emission",
            "categories": ("air",),
        },
        ("toy-biosphere", "ch4"): {
            "name": "Methane, fossil",
            "unit": "kilogram",
            "type": "emission",
            "categories": ("air",),
        },
    }
)
co2, ch4 = ("toy-biosphere", "co2"), ("toy-biosphere", "ch4")
bg_name = "toy-background"
bd.Database(bg_name).write(
    {
        (bg_name, "electricity"): {
            "name": "electricity production",
            "reference product": "electricity",
            "unit": "kilowatt hour",
            "location": "GLO",
            "exchanges": [
                {"input": (bg_name, "electricity"), "amount": 1, "type": "production"},
                {"input": co2, "amount": 0.5, "type": "biosphere"},
                {"input": ch4, "amount": 0.001, "type": "biosphere"},
            ],
        },
        (bg_name, "fabric"): {
            "name": "cotton fabric production",
            "reference product": "cotton fabric",
            "unit": "kilogram",
            "location": "GLO",
            "exchanges": [
                {"input": (bg_name, "fabric"), "amount": 1, "type": "production"},
                {"input": (bg_name, "electricity"), "amount": 3, "type": "technosphere"},
                {"input": co2, "amount": 2.0, "type": "biosphere"},
                {"input": ch4, "amount": 0.004, "type": "biosphere"},
            ],
        },
        (bg_name, "incineration"): {
            "name": "incineration of textile waste",
            "reference product": "textile waste",
            "unit": "kilogram",
            "location": "GLO",
            "exchanges": [
                {"input": (bg_name, "incineration"), "amount": -1, "type": "production"},
                {"input": co2, "amount": 1.2, "type": "biosphere"},
            ],
        },
    }
)
method = ("toy", "climate change")
bd.Method(method).write([(co2, 1.0), (ch4, 29.8)])
print(bd.databases)
print(bd.methods)

Databases dictionary with 2 object(s):
	toy-background
	toy-biosphere
Methods dictionary with 1 object(s):
	('toy', 'climate change')


**headwater.** The same background was imported once and frozen as a release, `toy-background@6864b7dbb65d`, with its method inside. You pull it into the cache once, then load it. `hw.load` checks every file against its hash, builds the matrices and solves the score of every product, once.

In [4]:
headwater("release", "pull", demo / "releases/toy-background@6864b7dbb65d")
bg = hw.load("toy-background@6864b7dbb65d")
print(bg)

Pulled release toy-background@6864b7dbb65d into <work>/headwater-home/releases
Background(release toy-background@6864b7dbb65d, method toy-climate 1, 1 indicator, 3 processes)


**What differs.**

- **Once, and frozen.** In Brightway the database lives in the project and can be changed by any later `write` or `save`. A headwater release is never edited: its name holds a hash of its content, every database pins it in `database.yaml`, and a result records it. Importing the same files again gives the same hash.
- **Rewrites.** `Database.write` deletes the database and writes it again, with new internal ids. [docs/findings.md](../docs/findings.md) shows a case where rewriting the emissions database with identical data took this T-shirt from 1.19205 to 0, because the method still pointed at the old ids. headwater never rewrites a release, and ids are text that never changes.
- **The method travels with the data.** Brightway's methods are stored in the project, apart from the databases. A headwater release holds its methods, and `hw.load` can also take method folders from elsewhere.
- **Names.** A unit is written as its source writes it: `kg`, `kWh` and ecoinvent's `metric ton*km`, where Brightway says `kilogram`, `kilowatt hour` and `ton kilometer` ([calculation.md](../docs/calculation.md#names-of-units-and-indicators) lists them all). Brightway's method `('ecoinvent-3.12', 'EF v3.1', 'climate change', 'global warming potential (GWP100)')` is the method `EF-3.1` and its indicator `climate-change`: the indicator's `name` is the impact category, `climate change`, and its `category_indicator` the last part, as ecoinvent's workbook gives it.

## 3. Find an activity

**Brightway.** `Database.search` asks a full-text index, which bw2data 4 keeps in an SQLite FTS5 database beside the data, and ranks what it finds. `get_node` looks up one activity by its fields.

In [5]:
background_db = bd.Database(bg_name)
print("search('fabric'):           ", background_db.search("fabric"))
print("search('production cotton'):", background_db.search("production cotton"))
print("search('fab'):              ", background_db.search("fab"))
fabric = bd.get_node(database=bg_name, name="cotton fabric production", location="GLO")
print("get_node:", fabric, fabric.key)

search('fabric'):            ['cotton fabric production' (kilogram, GLO, None)]
search('production cotton'): ['cotton fabric production' (kilogram, GLO, None)]
search('fab'):               []
get_node: 'cotton fabric production' (kilogram, GLO, None) ('toy-background', 'fabric')


**headwater.** `bg.find` checks the process names, then the product names, against six rules in a fixed order, and `why` says which one matched. A word may be part of a word, so `fab` finds fabric. There is no index file: the table is built in memory on the first call.

In [6]:
for text in ["fabric", "production cotton", "fab"]:
    for match in bg.find(text):
        print(f"find({text!r}): {match.process_id}  ({match.why})")

find('fabric'): toy:fabric-production  (every word in name)
find('production cotton'): toy:fabric-production  (every word in name)
find('fab'): toy:fabric-production  (every word in name)


A place that no process has is an error with a suggestion, not an empty list:

In [7]:
try:
    bg.find("fabric", location="GL0")
except ValueError as error:
    print(error)

no process is in 'GL0'. Did you mean 'GLO'? Places: GLO


**What differs.** Brightway's search is a ranked text search over an index stored next to the data. It is forgiving about word order, and it ranks large result lists well. headwater's `find` is exact: no ranking by relevance, no synonyms, no spelling repair, and the same text always gives the same list, with the rule that matched. Since a release never changes, there is no index to keep in step with the data. In Brightway an activity is known by its key `(database, code)`. In headwater it is known by its id, which holds the source and version for a release: `toy:fabric-production` here, `ecoinvent:3.12:cutoff:<activity uuid>_<product uuid>` for ecoinvent.

## 4. One footprint

The T-shirt buys 0.25 kg of cotton fabric and 0.5 kWh of electricity.

**Brightway.** The T-shirt is an activity in a foreground database. `bc.LCA` takes a demand and a method; `lci()` solves the inventory and `lcia()` characterises it.

In [8]:
bd.Database("tshirt").write(
    {
        ("tshirt", "tshirt"): {
            "name": "T-shirt",
            "reference product": "T-shirt",
            "unit": "unit",
            "location": "GLO",
            "exchanges": [
                {"input": ("tshirt", "tshirt"), "amount": 1, "type": "production"},
                {"input": (bg_name, "fabric"), "amount": 0.25, "type": "technosphere"},
                {"input": (bg_name, "electricity"), "amount": 0.5, "type": "technosphere"},
            ],
        }
    }
)
tshirt = bd.get_node(database="tshirt", code="tshirt")
lca = bc.LCA({tshirt: 1}, method=method)
lca.lci()
lca.lcia()
print(f"Brightway: {lca.score:.10f}")

Brightway: 1.1920500015


**headwater.** The T-shirt is a database folder: a `database.yaml` and two tables, `processes.csv` with one row per process and `exchanges.csv` with one row per line. `hw.open` reads it. `calc` scores one unit of what the process `tshirt:tshirt` makes, as `bc.LCA({tshirt: 1})` names the activity, with no new solve of the background.

In [9]:
database_folder = demo / "studies/tshirt"
print((database_folder / "exchanges.csv").read_text())
result = hw.open(database_folder).calc(bg, "tshirt:tshirt")
print(f"headwater: {result.score:.10f}")

assert abs(lca.score - 1.19205) < 1e-6 * 1.19205
assert abs(result.score - 1.19205) < 1e-9 * 1.19205

process,line,role,flow,amount,unit,supplier
tshirt,e01,reference,tshirt:tshirt,1,unit,
tshirt,e02,input,toy:cotton-fabric,0.25,kg,toy:fabric-production
tshirt,e03,input,toy:electricity,0.5,kWh,toy:electricity-production

headwater: 1.1920500000


Both give 1.19205. The digits after the seventh differ because Brightway stores every amount and factor as a 32-bit float, about 7 significant digits, and headwater as a 64-bit float. On ecoinvent 3.12 that difference is at most 1.7 × 10⁻⁷ relative, for every product ([benchmarks](../docs/appendix/benchmarks.md#parity-with-brightway)).

**Validation before calculation.** Add an emission the method does not count, 0.001 kg of nitrous oxide, to the T-shirt in both tools. In Brightway it counts 0, and nothing says so:

In [10]:
n2o = bd.Database("toy-biosphere").new_node(
    code="n2o", name="Dinitrogen monoxide", unit="kilogram", type="emission", categories=("air",)
)
n2o.save()
tshirt.new_edge(input=n2o, amount=0.001, type="biosphere").save()
lca = bc.LCA({tshirt: 1}, method=method)
lca.lci()
lca.lcia()
print(f"Brightway with N2O: {lca.score:.5f}")

Brightway with N2O: 1.19205


In headwater the same line stops the calculation, with a message that names the flow and says how to accept it: a row in the database's `no_factor.csv`, with a reason, who and when.

In [11]:
with_n2o = work / "tshirt-n2o/tshirt"
shutil.copytree(database_folder, with_n2o)
with (with_n2o / "flows.csv").open("a") as file:
    file.write("tshirt:n2o,Dinitrogen monoxide,elementary,kg,air,\n")
with (with_n2o / "exchanges.csv").open("a") as file:
    file.write("tshirt,e04,emission,tshirt:n2o,0.001,kg,\n")
try:
    hw.open(with_n2o).calc(bg, "tshirt:tshirt")
except hw.InvalidDataError as error:
    print(error)

1 problem:
- flow tshirt:n2o: has no factor in method toy-climate; used by tshirt:tshirt. To count it as 0, add a row to the database's no_factor.csv: flow, method, reason, who, when.


**What differs.** Brightway calculates whatever is linked. headwater checks the database first, with every problem in one message: unknown units, missing suppliers, a supplier that makes something else, a flow with no factor. `db.validate(bg)` runs the same checks without calculating. A flow of the **background** that the method does not count is not an error, since ecoinvent has many: it is listed on `result.not_characterised`.

## 5. Every footprint

**Brightway.** Factorize the matrix once with `lci(factorize=True)`, then `redo_lcia` with a new demand for each activity. `MultiLCA` does many demands and methods in one object.

In [12]:
lca = bc.LCA({fabric: 1}, method=method)
lca.lci(factorize=True)
lca.lcia()
brightway_scores = {}
for activity in sorted(bd.Database(bg_name), key=lambda activity: activity["name"]):
    lca.redo_lcia({activity.id: 1})
    brightway_scores[activity["name"]] = lca.score
for name, score in brightway_scores.items():
    print(f"{name:32} {score:9.5f}")

cotton fabric production           3.70860
electricity production             0.52980
incineration of textile waste     -1.20000


**headwater.** `hw.load` already solved the score of every product, for every indicator. `bg.score(process_id)` reads one, as `lca.score` does after `redo_lcia`, with no solve. `hw.score_many` reads many at once, for a table of products and amounts, as Brightway's `MultiLCA` does.

In [13]:
for process_id in bg.release.table("processes")["id"].to_pylist():
    print(f"{process_id:32} {bg.score(process_id):9.5f}")

toy:electricity-production         0.52980
toy:fabric-production              3.70860
toy:incineration-textile-waste     1.20000


**What differs.** Two things.

- **Speed.** Brightway solves once per product, reusing the factorization. headwater solves the transposed system once per indicator, which gives every product's score at once. [Notebook 05](05-brightway-and-headwater-on-ecoinvent.ipynb) times both on ecoinvent.
- **Waste.** For the incineration Brightway gives -1.2, and headwater +1.2. Both are right in their own terms. Brightway follows ecoinvent's convention, where a treatment's reference is -1 kg, so a demand of +1 is minus one kg treated. headwater reports a treatment per kg treated, as a positive burden, so a SKU or a database line multiplies kg sent by +1.2 whatever the product is. Inside the matrix headwater uses ecoinvent's convention too ([docs/design.md](../docs/design.md#roles-and-signs)).

## 6. Contribution analysis

Which processes emit the T-shirt's score? First restore the T-shirt without N₂O in Brightway, by setting that exchange to 0.

**Brightway.** `lca.characterized_inventory` is a matrix of flows by activities. Summing it by activity gives what each activity emits for the T-shirt. bw2analyzer's `ContributionAnalysis` does this and sorts it.

In [14]:
n2o_edge = next(e for e in tshirt.biosphere() if e.input["code"] == "n2o")
n2o_edge.delete()

lca = bc.LCA({tshirt: 1}, method=method)
lca.lci()
lca.lcia()
by_activity = np.asarray(lca.characterized_inventory.sum(axis=0)).ravel()
activity_of = {column: node_id for node_id, column in lca.dicts.activity.items()}
for column in sorted(range(len(by_activity)), key=lambda c: -abs(by_activity[c])):
    if by_activity[column]:
        node = bd.get_node(id=activity_of[column])
        print(f"{node['name']:28} {node['location']}  {by_activity[column]:.5f}")

electricity production       GLO  0.66225
cotton fabric production     GLO  0.52980


**headwater.** `result.where()` gives the same split, with how much each process runs and a sum by location. `result.contributions()` gives the other view: what the T-shirt buys, and what each purchase is worth.

In [15]:
print(result.where())
print()
print(result.contributions())

1 unit of T-shirt: 1.19205
By process:
  1.25 kWh of electricity production (GLO): 0.66225 (55.6%)
  0.25 kg of cotton fabric production (GLO): 0.52980 (44.4%)
By location:
  GLO: 1.19205 (100.0%), 2 processes

1 unit of T-shirt: 1.19205 (100.0%), direct 0.00000
  0.25 kg of cotton fabric production: 0.92715 (77.8%), direct 0.52980
    0.75 kWh of electricity production: 0.39735 (33.3%), direct 0.39735
  0.5 kWh of electricity production: 0.26490 (22.2%), direct 0.26490


**What differs.** The numbers are the same: electricity emits 0.66225 and fabric 0.52980. headwater adds the tree of purchases, the split by location, and the same three views for one unit of any background product, with no database of your own (`bg.where`, `bg.contributions`, `bg.sensitivity`). In Brightway the tree of purchases comes from a graph traversal library, which the pinned versions here do not include. On 10 products of ecoinvent 3.12 and 2 indicators, `bg.where` lists the same top 10 processes as bw2analyzer, in the same order, in 20 of 20 checks ([benchmarks](../docs/appendix/benchmarks.md#top-contributors-against-brightway)).

## 7. Change one input and recompute

Make the T-shirt heavier: 0.30 kg of fabric instead of 0.25.

**Brightway.** Find the exchange, change its amount, save it, and calculate again. The new amount is written into the project's SQLite database, and the old one is gone.

In [16]:
edge = next(e for e in tshirt.technosphere() if e.input["name"] == "cotton fabric production")
edge["amount"] = 0.30
edge.save()
lca = bc.LCA({tshirt: 1}, method=method)
lca.lci()
lca.lcia()
print(f"Brightway: {lca.score:.5f}")

Brightway: 1.37748


**headwater.** First ask what the change will do. `result.sensitivity()` gives the sensitivity of every line: the change of the score per one more unit. For the fabric line it is the score of 1 kg of fabric, 3.7086, so 0.05 kg more should add 0.18543.

In [17]:
fabric_line = next(line for line in result.sensitivity() if line.line_id == "e02")
print(
    f"line e02: sensitivity {fabric_line.sensitivity:.5f}, elasticity {fabric_line.elasticity:.4f}"
)
predicted = result.score + fabric_line.sensitivity * 0.05
print(f"predicted: {predicted:.5f}")

line e02: sensitivity 3.70860, elasticity 0.7778
predicted: 1.37748


Then make the change: one row of `exchanges.csv`. In a git repository this is the one-line diff a reviewer sees.

In [18]:
heavier = work / "heavier/tshirt"
shutil.copytree(database_folder, heavier)
lines = heavier / "exchanges.csv"
lines.write_text(lines.read_text().replace(",0.25,", ",0.30,"))
heavier_result = hw.open(heavier).calc(bg, "tshirt:tshirt")
print(f"headwater: {heavier_result.score:.5f}")

assert abs(lca.score - 1.37748) < 1e-6 * 1.37748
assert abs(heavier_result.score - 1.37748) < 1e-9 * 1.37748

headwater: 1.37748


**What differs.**

- **History.** A Brightway edit changes the project in place. To compare before and after, you copy the project or keep notes. A headwater database is text in git: every change has an author and a reason, and `headwater diff` scores both versions and ranks the lines that changed. CI posts that on every pull request ([docs/workflows.md](../docs/workflows.md#ci)).
- **Sensitivity analysis.** headwater ranks every line of the database and of the background it reaches by elasticity, the % change of the score per 1% change of the line, from what one calculation already solved. bw2calc has no such ranking; you change values and recompute.
- **Scenarios.** To say "the grid emits 0.3 kg per kWh, for everything", Brightway users edit the database or copy it. headwater has two separate tools: an override changes a background process for one database only, and a named scenario of a study changes it for everything, in one calculation, without touching the release ([notebook 02](02-writing-a-database.ipynb)).

## 8. Import an Excel inventory

Many inventories are written in Brightway's Excel format. Here is the T-shirt in it, with the fabric amount as a project parameter and a waste line: 0.05 kg of scraps sent to incineration, written negative as Brightway does. The cell writes it to an `.xlsx` file with openpyxl.

In [19]:
import openpyxl  # noqa: E402

rows = [
    ["Database", "toy-foreground"],
    [],
    ["Project parameters"],
    ["name", "amount", "formula", "comment"],
    ["fabric_kg", 0.25, None, "Fabric in one T-shirt"],
    [],
    ["Activity", "T-shirt"],
    ["production amount", 1],
    ["reference product", "T-shirt"],
    ["location", "GLO"],
    ["unit", "unit"],
    ["Exchanges"],
    ["name", "amount", "unit", "location", "type", "database", "reference product", "formula"],
    ["T-shirt", 1, "unit", "GLO", "production", "toy-foreground", "T-shirt", None],
    [
        "cotton fabric production",
        0.25,
        "kilogram",
        "GLO",
        "technosphere",
        "toy-background",
        "cotton fabric",
        "fabric_kg",
    ],
    [
        "electricity production",
        0.5,
        "kilowatt hour",
        "GLO",
        "technosphere",
        "toy-background",
        "electricity",
        None,
    ],
    [
        "incineration of textile waste",
        -0.05,
        "kilogram",
        "GLO",
        "technosphere",
        "toy-background",
        "textile waste",
        None,
    ],
]
workbook = openpyxl.Workbook()
sheet = workbook.active
sheet.title = "inventory"
for row in rows:
    sheet.append(row)
xlsx = work / "tshirt.xlsx"
workbook.save(xlsx)

**Brightway.** `ExcelImporter` reads the workbook. Its strategies normalise it, `match_database` links each line to an activity by the fields you name, `statistics` says what is still unlinked, and `write_database` writes it into the project. Reading the workbook and the strategies print a time, so the cell hides that part.

In [20]:
with contextlib.redirect_stdout(io.StringIO()):
    importer = bi.ExcelImporter(xlsx)
    importer.apply_strategies()

    fields = ("name", "reference product", "unit", "location")
    importer.match_database(bg_name, fields=fields)
    importer.match_database(fields=fields)
importer.statistics()
importer.write_project_parameters()
importer.write_database(activate_parameters=True)

imported = bd.get_node(database="toy-foreground", name="T-shirt")
lca = bc.LCA({imported: 1}, method=method)
lca.lci()
lca.lcia()
print(f"Brightway: {lca.score:.5f}")

Graph statistics for `toy-foreground` importer:
1 graph nodes:
	None: 1
4 graph edges:
	technosphere: 3
	production: 1
4 edges to the following databases:
	toy-background: 3
	toy-foreground: 1
0 unique unlinked edges (0 total):


Created database: toy-foreground
Brightway: 1.25205


**headwater.** `headwater database import-excel` reads the same workbook, links each line to a process of the release by exact name, reference product and location, and writes a database folder. It prints a report of what mapped.

In [21]:
headwater(
    "database",
    "import-excel",
    xlsx,
    "--release",
    "toy-background@6864b7dbb65d",
    "--out",
    work / "databases/tshirt",
    "--name",
    "tshirt",
)

Read tshirt.xlsx: database toy-foreground, 1 activities, 4 exchange rows, 1 parameter rows.
Linked to release toy-background@6864b7dbb65d. Database tshirt.

Lines
  mapped             4  (reference 1, release 3)
  unmapped           0
  waiting            0
  unexpressible      0

Databases named by the lines
  toy-background: 3 lines, the background, release toy-background

Parameters: 1 copied, 0 not copied

Wrote database tshirt to <work>/databases/tshirt
Validation: 0 errors, 0 warnings.


The database is ordinary files. Its process id, the `id` of its row in `processes.csv`, comes from the activity's name and location: `tshirt:t-shirt-glo`. The waste line became a positive 0.05 kg with the role `waste_to_treatment`, decided by what the supplier makes, not by the sign. The formula became a parameter.

In [22]:
excel_folder = work / "databases/tshirt"
print((excel_folder / "exchanges.csv").read_text())
print((excel_folder / "parameters.yaml").read_text())
excel_result = hw.open(excel_folder).calc(bg, "tshirt:t-shirt-glo")
print(f"headwater: {excel_result.score:.5f}")

assert abs(lca.score - 1.25205) < 1e-6 * 1.25205
assert abs(excel_result.score - 1.25205) < 1e-9 * 1.25205

process,line,role,flow,amount,unit,supplier,flow_name
t-shirt-glo,e01,reference,tshirt:t-shirt-glo,1,unit,,T-shirt
t-shirt-glo,e02,input,toy:cotton-fabric,fabric_kg,kg,toy:fabric-production,cotton fabric
t-shirt-glo,e03,input,toy:electricity,0.5,kWh,toy:electricity-production,electricity
t-shirt-glo,e04,waste_to_treatment,toy:textile-waste,0.05,kg,toy:incineration-textile-waste,textile waste

database:
  fabric_kg:
    value: 0.25
    comment: Fabric in one T-shirt

headwater: 1.25205


**What differs.** Both give 1.25205: the T-shirt plus 0.05 kg × 1.2 for the waste. Brightway writes the result into the project, once every line is linked. headwater writes a database folder in any case: a line it cannot link keeps the workbook's own words in a comment, has no supplier, and `validate` refuses the database until a person decides. That decision goes in a mapping file, with who and when, kept in git and reused at the next import. `database.yaml` gets the name and the release, and the method when you give one with `--method`. The products to score, and how much of each, go in a study next to the database ([notebook 02](02-writing-a-database.ipynb)).

## 9. Import a Brightway project

Your inventory may already be in a Brightway project. `headwater database import-brightway` turns one database of it into a headwater database folder, from the project's own files: it does not need bw2data, and it only reads them. Each exchange is linked by the key of the node it comes from, so a line goes to the activity Brightway linked it to. Parameters, formulas and distributions come along.

Here it reads the toy project that [bench/bw_toy_project.py](../bench/bw_toy_project.py) wrote with bw2data 4.5.4. Its database `tshirt` is the T-shirt of section 8, with the electricity line lognormal, so it scores the same 1.25205.

In [23]:
shutil.copytree(repo / "tests/fixtures/brightway", work / "projects")
headwater(
    "database",
    "import-brightway",
    "toy",
    "tshirt",
    "--brightway-dir",
    work / "projects",
    "--release",
    "toy-background@6864b7dbb65d",
    "--out",
    work / "from-project/tshirt",
)
project_folder = work / "from-project/tshirt"
print((project_folder / "exchanges.csv").read_text())
project_result = hw.open(project_folder).calc(bg, "tshirt:t-shirt-glo")
print(f"headwater: {project_result.score:.5f}")

assert abs(project_result.score - 1.25205) < 1e-9 * 1.25205

Read toy.10016b6e: database tshirt, 1 activities, 4 exchange rows, 3 parameter rows.
Linked to release toy-background@6864b7dbb65d. Database tshirt.

Lines
  mapped             4  (reference 1, release 3)
  unmapped           0
  waiting            0
  unexpressible      0

Databases named by the lines
  toy-background: 3 lines, the background, release toy-background

Parameters: 3 copied, 0 not copied

Wrote database tshirt to <work>/from-project/tshirt
Validation: 0 errors, 0 warnings.
process,line,role,flow,amount,unit,supplier,flow_name,comment,distribution,scale
t-shirt-glo,e01,reference,tshirt:t-shirt-glo,1,unit,,T-shirt,,,
t-shirt-glo,e02,input,toy:cotton-fabric,fabric_kg,kg,toy:fabric-production,cotton fabric,Fabric in the bill of materials,,
t-shirt-glo,e03,input,toy:electricity,0.5,kWh,toy:electricity-production,electricity,,lognormal,0.09531017980432493
t-shirt-glo,e04,waste_to_treatment,toy:textile-waste,-(-waste_kg),kg,toy:incineration-textile-waste,textile waste,Scraps se

## What Brightway has that headwater does not, yet

- **Other formats.** bw2io also reads EXIOBASE. headwater reads ecoinvent's ecoSpold2, ecoSpold1, openLCA JSON-LD, SimaPro CSV and ecoinvent's LCIA workbook; ILCD is on the roadmap's Later list.
- **Regionalised and temporal LCA**, through Brightway's extra packages.
- **A graphical interface**: the Activity Browser.
- **Computed allocation for your own multi-output processes.** A headwater database process makes one product. Splitting a process that makes several is done by the importers, from the source's own factors.

And Brightway has a large community, years of use in research, and many packages built on it.

## Next

[05-brightway-and-headwater-on-ecoinvent](05-brightway-and-headwater-on-ecoinvent.ipynb) runs these tasks on ecoinvent 3.12 and times them. It needs an ecoinvent licence.